In [9]:
%%writefile cuda_crosscorr.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define N 65536

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = (call); \
        if (err != cudaSuccess) { \
            fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(err)); \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

__global__ void crosscorr_kernel(const float * __restrict__ x,
                                  const float * __restrict__ y,
                                  float * __restrict__ r, int n) {
    int lag = blockIdx.x * blockDim.x + threadIdx.x;
    if (lag >= n) return;
    float sum = 0.0f;
    for (int i = 0; i < n - lag; i++) {
        sum += __ldg(&x[i]) * __ldg(&y[i + lag]);
    }
    r[lag] = sum;
}

void init_signals(float *x, float *y, int n) {
    FILE *fx = fopen("signal_x.csv", "r");
    FILE *fy = fopen("signal_y.csv", "r");
    if (!fx || !fy) {
        fprintf(stderr, "ERROR: Cannot open signal CSV files\n");
        exit(1);
    }
    for (int i = 0; i < n; i++) {
        fscanf(fx, "%f", &x[i]);
        fscanf(fy, "%f", &y[i]);
    }
    fclose(fx);
    fclose(fy);
}

int main(int argc, char *argv[]) {
    int block_size = 512;
    if (argc >= 2) block_size = atoi(argv[1]);
    int grid_size = (N + block_size - 1) / block_size;

    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);
    printf("  Cross-Correlation -- CUDA Parallel\n");
    printf("  GPU: %s\n", prop.name);
    printf("  N = %d | Block size = %d | Grid size = %d\n",
           N, block_size, grid_size);
    printf("-----------------------------------------------\n\n");

    float *h_x = (float *)malloc(N * sizeof(float));
    float *h_y = (float *)malloc(N * sizeof(float));
    float *h_r = (float *)malloc(N * sizeof(float));

    init_signals(h_x, h_y, N);

    float *d_x, *d_y, *d_r;
    CUDA_CHECK(cudaMalloc(&d_x, N * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_y, N * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_r, N * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_x, h_x, N * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_y, h_y, N * sizeof(float), cudaMemcpyHostToDevice));

    /* Warm-up */
    crosscorr_kernel<<<grid_size, block_size>>>(d_x, d_y, d_r, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    /* Timed run */
    cudaEvent_t ev_start, ev_stop;
    CUDA_CHECK(cudaEventCreate(&ev_start));
    CUDA_CHECK(cudaEventCreate(&ev_stop));
    CUDA_CHECK(cudaEventRecord(ev_start));
    crosscorr_kernel<<<grid_size, block_size>>>(d_x, d_y, d_r, N);
    CUDA_CHECK(cudaEventRecord(ev_stop));
    CUDA_CHECK(cudaEventSynchronize(ev_stop));

    float elapsed_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&elapsed_ms, ev_start, ev_stop));
    CUDA_CHECK(cudaMemcpy(h_r, d_r, N * sizeof(float), cudaMemcpyDeviceToHost));

    printf("Execution time  : %.4f ms (%.6f s)\n",
           elapsed_ms, elapsed_ms / 1000.0f);
    printf("\n--- First 8 values (must match serial) ---\n");
    for (int k = 0; k < 8; k++)
        printf("  r[%5d] = %12.4f\n", k, h_r[k]);

    FILE *fp = fopen("cuda_timing.csv", "a");
    if (fp) { fprintf(fp, "%d,%d,%.4f\n", block_size, grid_size, elapsed_ms); fclose(fp); }
    printf("\nResult saved -> cuda_timing.csv\n");

    cudaFree(d_x); cudaFree(d_y); cudaFree(d_r);
    free(h_x); free(h_y); free(h_r);
    return 0;
}

Overwriting cuda_crosscorr.cu


In [10]:
from google.colab import files
files.upload()

Saving signal_x.csv to signal_x.csv


{'signal_x.csv': b'0.0\n0.0980171403295606\n0.19509032201612825\n0.29028467725446233\n0.3826834323650898\n0.47139673682599764\n0.5555702330196022\n0.6343932841636455\n0.7071067811865475\n0.773010453362737\n0.8314696123025452\n0.8819212643483549\n0.9238795325112867\n0.9569403357322089\n0.9807852804032304\n0.9951847266721968\n1.0\n0.9951847266721969\n0.9807852804032304\n0.9569403357322089\n0.9238795325112867\n0.881921264348355\n0.8314696123025455\n0.7730104533627371\n0.7071067811865476\n0.6343932841636455\n0.5555702330196022\n0.47139673682599786\n0.3826834323650899\n0.2902846772544624\n0.1950903220161286\n0.09801714032956083\n1.2246467991473532e-16\n-0.09801714032956059\n-0.19509032201612836\n-0.2902846772544621\n-0.38268343236508967\n-0.47139673682599764\n-0.555570233019602\n-0.6343932841636453\n-0.7071067811865475\n-0.7730104533627367\n-0.8314696123025452\n-0.8819212643483549\n-0.9238795325112865\n-0.9569403357322088\n-0.9807852804032303\n-0.9951847266721969\n-1.0\n-0.9951847266721969\

In [11]:
from google.colab import files
files.upload()

Saving signal_y.csv to signal_y.csv


{'signal_y.csv': b'0.0\n0.19509032201612825\n0.3826834323650898\n0.5555702330196022\n0.7071067811865475\n0.8314696123025452\n0.9238795325112867\n0.9807852804032304\n1.0\n0.9807852804032304\n0.9238795325112867\n0.8314696123025455\n0.7071067811865476\n0.5555702330196022\n0.3826834323650899\n0.1950903220161286\n1.2246467991473532e-16\n-0.19509032201612836\n-0.38268343236508967\n-0.555570233019602\n-0.7071067811865475\n-0.8314696123025452\n-0.9238795325112865\n-0.9807852804032303\n-1.0\n-0.9807852804032304\n-0.9238795325112866\n-0.8314696123025455\n-0.7071067811865477\n-0.5555702330196022\n-0.3826834323650904\n-0.19509032201612872\n-2.4492935982947064e-16\n0.19509032201612825\n0.38268343236508995\n0.5555702330196018\n0.7071067811865474\n0.8314696123025452\n0.9238795325112865\n0.9807852804032303\n1.0\n0.9807852804032307\n0.9238795325112867\n0.8314696123025456\n0.7071067811865483\n0.5555702330196023\n0.3826834323650905\n0.19509032201612797\n3.6739403974420594e-16\n-0.19509032201612725\n-0.38

In [13]:
!nvcc -O3 -arch=sm_75 -o cuda_crosscorr cuda_crosscorr.cu


cuda_crosscorr.cu(37): warning #1650-D: result of call is not used
          fscanf(fx, "%f", &x[i]);
          ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

cuda_crosscorr.cu(38): warning #1650-D: result of call is not used
          fscanf(fy, "%f", &y[i]);
          ^

cuda_crosscorr.cu(37): warning #1650-D: result of call is not used
          fscanf(fx, "%f", &x[i]);
          ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

cuda_crosscorr.cu(38): warning #1650-D: result of call is not used
          fscanf(fy, "%f", &y[i]);
          ^

cuda_crosscorr.cu: In function ‘void init_signals(float*, float*, int)’:
cuda_crosscorr.cu:37:7: warning: ignoring return value of ‘int fscanf(FILE*, const char*, ...)’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   37 |         fscanf(fx, "%f", &x[i]);
      |       ^ ~~~~~

In [14]:
!echo "block_size,grid_size,time_ms" > cuda_timing.csv
!./cuda_crosscorr 32
!./cuda_crosscorr 64
!./cuda_crosscorr 128
!./cuda_crosscorr 256
!./cuda_crosscorr 512
!./cuda_crosscorr 1024

  Cross-Correlation -- CUDA Parallel
  GPU: Tesla T4
  N = 65536 | Block size = 32 | Grid size = 2048

Execution time  : 10.5358 ms (0.010536 s)

--- First 8 values (must match serial) ---
  r[    0] =      -0.0000
  r[    1] =      -0.0010
  r[    2] =       0.0191
  r[    3] =       0.0746
  r[    4] =       0.1848
  r[    5] =       0.3644
  r[    6] =       0.6191
  r[    7] =       0.9594

Result saved -> cuda_timing.csv
  Cross-Correlation -- CUDA Parallel
  GPU: Tesla T4
  N = 65536 | Block size = 64 | Grid size = 1024

Execution time  : 9.8366 ms (0.009837 s)

--- First 8 values (must match serial) ---
  r[    0] =      -0.0000
  r[    1] =      -0.0010
  r[    2] =       0.0191
  r[    3] =       0.0746
  r[    4] =       0.1848
  r[    5] =       0.3644
  r[    6] =       0.6191
  r[    7] =       0.9594

Result saved -> cuda_timing.csv
  Cross-Correlation -- CUDA Parallel
  GPU: Tesla T4
  N = 65536 | Block size = 128 | Grid size = 512

Execution time  : 8.1506 ms (0.008151 

In [4]:
!echo "block_size,grid_size,time_ms" > cuda_timing.csv
!./cuda_crosscorr 32
!./cuda_crosscorr 64
!./cuda_crosscorr 128
!./cuda_crosscorr 256
!./cuda_crosscorr 512
!./cuda_crosscorr 1024

  Cross-Correlation -- CUDA Parallel
  GPU: Tesla T4
  N = 65536 | Block size = 32 | Grid size = 2048

Execution time  : 7.5667 ms (0.007567 s)

--- First 8 values (must match serial) ---
  r[    0] =       0.0000
  r[    1] =   -3212.0464
  r[    2] =   -6393.6802
  r[    3] =   -9511.7432
  r[    4] =  -12539.8105
  r[    5] =  -15447.7109
  r[    6] =  -18197.2461
  r[    7] =  -20783.2031

Result saved -> cuda_timing.csv
  Cross-Correlation -- CUDA Parallel
  GPU: Tesla T4
  N = 65536 | Block size = 64 | Grid size = 1024

Execution time  : 7.4388 ms (0.007439 s)

--- First 8 values (must match serial) ---
  r[    0] =       0.0000
  r[    1] =   -3212.0464
  r[    2] =   -6393.6802
  r[    3] =   -9511.7432
  r[    4] =  -12539.8105
  r[    5] =  -15447.7109
  r[    6] =  -18197.2461
  r[    7] =  -20783.2031

Result saved -> cuda_timing.csv
  Cross-Correlation -- CUDA Parallel
  GPU: Tesla T4
  N = 65536 | Block size = 128 | Grid size = 512

Execution time  : 7.7700 ms (0.007770 s

In [15]:
from google.colab import files
files.download('cuda_timing.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
%%writefile cuda_autocorr.cu

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define N 65536

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = (call); \
        if (err != cudaSuccess) { \
            fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(err)); \
            exit(EXIT_FAILURE); \
        } \
    } while (0)

__global__ void autocorr_kernel(const float * __restrict__ x,
                                 float * __restrict__ r, int n) {
    int lag = blockIdx.x * blockDim.x + threadIdx.x;
    if (lag >= n) return;
    float sum = 0.0f;
    for (int i = 0; i < n - lag; i++) {
        sum += __ldg(&x[i]) * __ldg(&x[i + lag]);
    }
    r[lag] = sum;
}

void init_signal(float *x, int n) {
    for (int i = 0; i < n; i++) {
        x[i] = (float)sin(2.0 * M_PI * (double)i / 64.0);
    }
}

int main(int argc, char *argv[]) {
    int block_size = 512;
    if (argc >= 2) block_size = atoi(argv[1]);
    int grid_size = (N + block_size - 1) / block_size;

    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);
    printf("  Autocorrelation -- CUDA Parallel\n");
    printf("  GPU: %s\n", prop.name);
    printf("  N = %d | Block size = %d | Grid size = %d\n",
           N, block_size, grid_size);
    printf("---------------------------------------------------\n\n");

    float *h_x = (float *)malloc(N * sizeof(float));
    float *h_r = (float *)malloc(N * sizeof(float));

    init_signal(h_x, N);

    float *d_x, *d_r;
    CUDA_CHECK(cudaMalloc(&d_x, N * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_r, N * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_x, h_x, N * sizeof(float), cudaMemcpyHostToDevice));

    /* Warm-up */
    autocorr_kernel<<<grid_size, block_size>>>(d_x, d_r, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    /* Timed run */
    cudaEvent_t ev_start, ev_stop;
    CUDA_CHECK(cudaEventCreate(&ev_start));
    CUDA_CHECK(cudaEventCreate(&ev_stop));

    CUDA_CHECK(cudaEventRecord(ev_start));
    autocorr_kernel<<<grid_size, block_size>>>(d_x, d_r, N);
    CUDA_CHECK(cudaEventRecord(ev_stop));
    CUDA_CHECK(cudaEventSynchronize(ev_stop));

    float elapsed_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&elapsed_ms, ev_start, ev_stop));

    CUDA_CHECK(cudaMemcpy(h_r, d_r, N * sizeof(float), cudaMemcpyDeviceToHost));

    printf("Execution time  : %.4f ms (%.6f s)\n",
           elapsed_ms, elapsed_ms / 1000.0f);
    printf("\n--- First 8 values (must match serial) ---\n");
    for (int k = 0; k < 8; k++)
        printf("  r[%5d] = %12.4f\n", k, h_r[k]);

    FILE *fp = fopen("cuda_autocorr_timing.csv", "a");
    if (fp) { fprintf(fp, "%d,%d,%.4f\n", block_size, grid_size, elapsed_ms); fclose(fp); }
    printf("\nResult saved -> cuda_autocorr_timing.csv\n");

    cudaFree(d_x); cudaFree(d_r);
    free(h_x); free(h_r);
    return 0;
}

Writing cuda_autocorr.cu


In [7]:
!nvcc -O3 -arch=sm_75 -o cuda_autocorr cuda_autocorr.cu
!echo "block_size,grid_size,time_ms" > cuda_autocorr_timing.csv
!./cuda_autocorr 32
!./cuda_autocorr 64
!./cuda_autocorr 128
!./cuda_autocorr 256
!./cuda_autocorr 512
!./cuda_autocorr 1024

  Autocorrelation -- CUDA Parallel
  GPU: Tesla T4
  N = 65536 | Block size = 32 | Grid size = 2048

Execution time  : 10.4147 ms (0.010415 s)

--- First 8 values (must match serial) ---
  r[    0] =   32768.0000
  r[    1] =   32612.5293
  r[    2] =   32135.0352
  r[    3] =   31357.3496
  r[    4] =   30275.9727
  r[    5] =   28900.3008
  r[    6] =   27252.9668
  r[    7] =   25331.7031

Result saved -> cuda_autocorr_timing.csv
  Autocorrelation -- CUDA Parallel
  GPU: Tesla T4
  N = 65536 | Block size = 64 | Grid size = 1024

Execution time  : 8.5484 ms (0.008548 s)

--- First 8 values (must match serial) ---
  r[    0] =   32768.0000
  r[    1] =   32612.5293
  r[    2] =   32135.0352
  r[    3] =   31357.3496
  r[    4] =   30275.9727
  r[    5] =   28900.3008
  r[    6] =   27252.9668
  r[    7] =   25331.7031

Result saved -> cuda_autocorr_timing.csv
  Autocorrelation -- CUDA Parallel
  GPU: Tesla T4
  N = 65536 | Block size = 128 | Grid size = 512

Execution time  : 8.3890 m

In [8]:
from google.colab import files
files.download('cuda_autocorr_timing.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>